In [ ]:
import numpy as np
import re
import os
import json
from pathlib import Path

from gerrychain import Graph
from gerrychain.partition import recursive_tree_part

import networkx as nx
from networkx.readwrite import json_graph
import pandas as pd

from binary_ensemble.stream import BenDecoder
from gerrytools.mgrp import RunContainer, RecomRunnerConfig, RecomRunInfo

In [ ]:
CACHE_DIR = Path("chain_runs_frcw")
CACHE_DIR.mkdir(exist_ok=True)


def load_grid(filepath):
    with open(filepath, 'r') as file:
        lines = file.readlines()
    return lines


def effective_sample_size(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - x.mean()
    n = len(x)
    var = x.var()
    if var == 0:
        return n
    size = 1 << (2 * n - 1).bit_length()
    fx = np.fft.rfft(x, n=size)
    acov = np.fft.irfft(fx * np.conj(fx))[:n].real
    acf = acov / (var * n)
    rho_sum = 0.0
    for k in range(1, n // 2, 2):
        pair = acf[k] + acf[k + 1]
        if pair < 0:
            break
        rho_sum += pair
    return n / (1 + 2 * rho_sum)


def build_edges(rows, cols):
    edges = []
    for r in range(rows):
        for c in range(cols):
            node = r * cols + c
            if c < cols - 1:
                edges.append((node, node + 1))
            if r < rows - 1:
                edges.append((node, node + cols))
    edges = np.array(edges)
    return edges[:, 0], edges[:, 1]


def cut_edges(a, U, V):
    return int(np.sum(a[U] != a[V]))


def parse(line):
    return np.array([int(x) for x in line.strip().split(',')], dtype=int)


def get_cut_edges(assignments, rows, cols):
    U, V = build_edges(rows, cols)
    return [cut_edges(parse(line), U, V) for line in assignments]


def district_nodes(a, d):
    return np.flatnonzero(a == d)


def induced_edges(U, V, nodes):
    keep = np.isin(U, nodes) & np.isin(V, nodes)
    return U[keep], V[keep]


def laplacian(nodes, U_sub, V_sub):
    index_of = {node: i for i, node in enumerate(nodes)}
    n = len(nodes)
    L = np.zeros((n, n), dtype=int)
    for u, v in zip(U_sub, V_sub):
        i, j = index_of[u], index_of[v]
        L[i, i] += 1
        L[j, j] += 1
        L[i, j] -= 1
        L[j, i] -= 1
    return L


def spanning_trees(nodes, U_sub, V_sub):
    if len(nodes) <= 1:
        return 1
    L = laplacian(nodes, U_sub, V_sub)
    minor = L[1:, 1:]
    return round(np.linalg.det(minor))  # Kirchhoff's theorem


def plan_spanning_trees(a, U, V):
    total = 1
    for d in np.unique(a):
        nodes = district_nodes(a, d)
        U_sub, V_sub = induced_edges(U, V, nodes)
        total *= spanning_trees(nodes, U_sub, V_sub)
    return total


def get_spanning_trees(assignments, rows, cols):
    U, V = build_edges(rows, cols)
    return [plan_spanning_trees(parse(line), U, V) for line in assignments]


def ground_truth_distribution(assignments, rows, cols):
    weights = np.array(get_spanning_trees(assignments, rows, cols), dtype=float)
    Z = weights.sum()
    return weights / Z


def uniform_distribution(assignments):
    n = len(assignments)
    return np.full(n, 1.0 / n)


def weighted_stat_distribution(stat_values, weights):
    dist = {}
    for val, w in zip(stat_values, weights):
        dist[val] = dist.get(val, 0.0) + w
    return dist


def build_plan_index(assignments, rows, cols):
    index = {}
    for i, line in enumerate(assignments):
        key = canonicalize(parse(line))
        index[key] = i
    return index


def canonicalize(a):
    mapping = {}
    next_label = 1
    canon = np.empty_like(a)
    for i, val in enumerate(a):
        if val not in mapping:
            mapping[val] = next_label
            next_label += 1
        canon[i] = mapping[val]
    return tuple(canon)


def ground_truth_stat_dist(stat_vals, weights, n_bins):
    dist = np.zeros(n_bins)
    for v, w in zip(stat_vals, weights):
        dist[v] += w
    return dist


def tv_at_checkpoints(values, ground_truth_dist, checkpoint_steps):
    # TV(p,q) = 1 - sum min(p,q); unvisited plans have p=0 so they never need touching
    visited = {}
    results = []
    pos = 0
    for step in checkpoint_steps:
        for v in values[pos:step]:
            visited[v] = visited.get(v, 0) + 1
        pos = step
        sum_min = sum(min(count / step, ground_truth_dist[x]) for x, count in visited.items())
        results.append(1 - sum_min)
    return results


def tv_between(a_counts, b_counts):
    a_dist, b_dist = a_counts / a_counts.sum(), b_counts / b_counts.sum()
    return 0.5 * np.sum(np.abs(a_dist - b_dist))


def wasserstein_between(a_counts, b_counts):
    a_dist, b_dist = a_counts / a_counts.sum(), b_counts / b_counts.sum()
    return np.sum(np.abs(np.cumsum(a_dist) - np.cumsum(b_dist)))


def counts_at_checkpoints(values, n_bins, checkpoint_steps):
    counts = np.zeros(n_bins)
    pos = 0
    snapshots = []
    for step in checkpoint_steps:
        counts += np.bincount(values[pos:step], minlength=n_bins)
        pos = step
        snapshots.append(counts.copy())
    return snapshots


def cumulative_stats(values):
    x = values.astype(np.float64)
    return np.cumsum(x), np.cumsum(x * x)


def gr_at_checkpoint(cum_a, cum_b, t):
    sum_a, sumsq_a = cum_a[0][t - 1], cum_a[1][t - 1]
    sum_b, sumsq_b = cum_b[0][t - 1], cum_b[1][t - 1]
    mean_a, mean_b = sum_a / t, sum_b / t
    var_a = (sumsq_a - sum_a ** 2 / t) / (t - 1)
    var_b = (sumsq_b - sum_b ** 2 / t) / (t - 1)
    overall_mean = (mean_a + mean_b) / 2
    B = t * ((mean_a - overall_mean) ** 2 + (mean_b - overall_mean) ** 2)
    W = (var_a + var_b) / 2
    var_hat = (t - 1) / t * W + B / t
    return np.sqrt(var_hat / W)


def pairwise_max_at_checkpoints(cut_edges_by_seed, n_bins, checkpoint_steps):
    seeds = list(cut_edges_by_seed.keys())
    snapshots = {s: counts_at_checkpoints(cut_edges_by_seed[s], n_bins, checkpoint_steps) for s in seeds}
    cum = {s: cumulative_stats(cut_edges_by_seed[s]) for s in seeds}

    tv_curve, w_curve, gr_curve = [], [], []
    for k, step in enumerate(checkpoint_steps):
        tv_vals, w_vals, gr_vals = [], [], []
        for i in range(len(seeds)):
            for j in range(i + 1, len(seeds)):
                a_c, b_c = snapshots[seeds[i]][k], snapshots[seeds[j]][k]
                tv_vals.append(tv_between(a_c, b_c))
                w_vals.append(wasserstein_between(a_c, b_c))
                gr_vals.append(gr_at_checkpoint(cum[seeds[i]], cum[seeds[j]], step))

        tv_curve.append(max(tv_vals))
        w_curve.append(max(w_vals))
        gr_curve.append(max(gr_vals))

    return tv_curve, w_curve, gr_curve


def first_crossing_step(curve, checkpoint_steps, threshold, consecutive=3):
    run_start = None
    run_len = 0
    for value, step in zip(curve, checkpoint_steps):
        if value < threshold:
            if run_len == 0:
                run_start = step
            run_len += 1
            if run_len >= consecutive:
                return run_start
        else:
            run_len = 0
            run_start = None
    return None


def convergence_steps(tv_curve, w_curve, gr_curve, checkpoint_steps, tv_thresh, w_thresh, gr_thresh):
    return {
        "tv": first_crossing_step(tv_curve, checkpoint_steps, tv_thresh),
        "wasserstein": first_crossing_step(w_curve, checkpoint_steps, w_thresh),
        "gelman_rubin": first_crossing_step(gr_curve, checkpoint_steps, gr_thresh),
    }


def worst_seed_plan_tv_crossing(plan_ids_by_seed, pi, checkpoint_steps, threshold):
    seed_curves = {
        seed: tv_at_checkpoints(idx, pi, checkpoint_steps)
        for seed, idx in plan_ids_by_seed.items()
    }
    worst_curve = [max(c[k] for c in seed_curves.values()) for k in range(len(checkpoint_steps))]
    crossing_step = first_crossing_step(worst_curve, checkpoint_steps, threshold)
    return crossing_step, worst_curve


def wasserstein_noise_floor(cut_pi, n_samples, n_trials=500, rng=None):
    rng = rng or np.random.default_rng()
    values = np.arange(len(cut_pi))
    gt_cdf = np.cumsum(cut_pi)
    floor = np.empty(n_trials)
    for i in range(n_trials):
        sample = rng.choice(values, size=n_samples, p=cut_pi)
        counts = np.bincount(sample, minlength=len(cut_pi)).astype(float)
        emp_cdf = np.cumsum(counts / counts.sum())
        floor[i] = np.sum(np.abs(emp_cdf - gt_cdf))
    return floor


def derived_cache_path(m, n, seed):
    return CACHE_DIR / f"recom_{m}x{n}_seed_{seed}_derived.npz"


def load_derived_cache(m, n, seed, total_steps, enum_filename):
    path = derived_cache_path(m, n, seed)
    if not path.exists():
        return None
    data = np.load(path, allow_pickle=True)
    if not (data['m'] == m and data['n'] == n and data['seed'] == seed
            and data['total_steps'] == total_steps and str(data['enum_filename']) == enum_filename):
        return None
    return data['cut_edges'], data['plan_ids'], int(data['accepted'])


def save_derived_cache(m, n, seed, total_steps, enum_filename, cut_edges_arr, plan_ids_arr, accepted):
    np.savez(derived_cache_path(m, n, seed), cut_edges=cut_edges_arr, plan_ids=plan_ids_arr,
             accepted=accepted, m=m, n=n, seed=seed, total_steps=total_steps,
             enum_filename=enum_filename)


def grid_to_json(m, n):
    nx_graph = nx.grid_2d_graph(m, n)
    nx.set_node_attributes(nx_graph, 1, "population")
    relabel = {(r, c): r * n + c for r in range(m) for c in range(n)}
    g_int = nx.relabel_nodes(nx_graph, relabel, copy=True)

    G = Graph.from_networkx(nx_graph.copy())
    init = recursive_tree_part(G, parts=range(m), pop_target=m, pop_col="population", epsilon=0.0)
    # frcw requires 1-indexed district labels (gerrychain's own parts=range(m) is
    # 0-indexed) - confirmed by testing: 0-indexed input raises
    # ErrAssignmentVectorNotOneIndexed.
    init_relabel = {relabel[k]: v + 1 for k, v in init.items()}
    nx.set_node_attributes(g_int, init_relabel, "assignment")

    path = CACHE_DIR / f"graph_{m}x{n}.json"
    with open(path, "w") as f:
        json.dump(json_graph.adjacency_data(g_int), f)
    return path


def run_seed_frcw(container, config, seed, m, total_steps):
    run_info = RecomRunInfo(
        pop_col="population",
        assignment_col="assignment",
        variant="R",
        balance_ub=1,
        n_steps=total_steps,
        pop_tol=0.0,
        n_threads=1,
        batch_size=1,
        writer="ben",
        rng_seed=seed,
    )
    container.run(run_info)

    output_path = Path(config.output_folder) / f"RecomR_assignment_{seed}_{total_steps}.jsonl.ben"
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(
            f"frcw produced no output for seed {seed} (m={m}, steps={total_steps}) - "
            f"check the log at {config.log_file(run_info)}"
        )
    return output_path


def run_all_seeds_frcw(m, n, total_steps, seeds):
    json_path = grid_to_json(m, n)
    config = RecomRunnerConfig(
        json_file_path=str(json_path),
        output_folder=str(CACHE_DIR / "output"),
        log_folder=str(CACHE_DIR / "logs"),
    )
    paths = {}
    with RunContainer(config) as rc:
        for seed in seeds:
            paths[seed] = run_seed_frcw(rc, config, seed, m, total_steps)
    return paths


def decode_frcw_output(path, plan_index, U, V):
    decoder = BenDecoder(path)
    n = len(decoder)
    plan_ids = np.empty(n, dtype=np.int32)
    dtype = np.uint8 if len(U) < 256 else np.uint16
    cut_edges_arr = np.empty(n, dtype=dtype)
    accepted = 0
    prev = None
    for i, a in enumerate(decoder):
        arr = np.asarray(a)
        plan_ids[i] = plan_index[canonicalize(arr)]
        cut_edges_arr[i] = cut_edges(arr, U, V)
        if prev is None or not np.array_equal(arr, prev):
            accepted += 1
        prev = arr
    return cut_edges_arr, plan_ids, accepted


In [ ]:
filename = 'enum_Any[4,4]_4_rc.txt'
# filename = 'enum_Any[5,5]_5_rc.txt'
# filename = 'enumerated_grids/enum_Any[6,6]_6_rc.txt'  # not in this repo

In [ ]:

assignments = load_grid(filename)
m, n = map(int, re.search(r'\[(\d+),(\d+)\]', filename).groups())

cut_vals = np.array(get_cut_edges(assignments, m, n))
n_bins = max(cut_vals) + 1
pi = ground_truth_distribution(assignments, m, n)
cut_pi = ground_truth_stat_dist(cut_vals, pi, n_bins)
unif = uniform_distribution(assignments)
ground_truth_cut = weighted_stat_distribution(cut_vals, pi)
uniform_cut_dist = weighted_stat_distribution(cut_vals, unif)
plan_index = build_plan_index(assignments, m, n)
U, V = build_edges(m, n)


In [ ]:
seeds = np.arange(1, 16, 1)
total_steps = 2_000_000
run_paths = run_all_seeds_frcw(m, n, total_steps, seeds)


In [ ]:
cut_edges_by_seed = {}
plan_ids_by_seed = {}
accepted_by_seed = {}
for seed in seeds:
    cached = load_derived_cache(m, n, seed, total_steps, filename)
    if cached is None:
        cut_edges_arr, plan_ids_arr, accepted = decode_frcw_output(
            run_paths[seed], plan_index, U, V
        )
        save_derived_cache(m, n, seed, total_steps, filename,
                            cut_edges_arr, plan_ids_arr, accepted)
    else:
        cut_edges_arr, plan_ids_arr, accepted = cached
    cut_edges_by_seed[seed] = cut_edges_arr
    plan_ids_by_seed[seed] = plan_ids_arr
    accepted_by_seed[seed] = accepted

for seed, ce in cut_edges_by_seed.items():
    print(seed, "fraction of steps with an actual change:", np.mean(np.diff(ce) != 0))


In [ ]:
n_bins = max(v.max() for v in cut_edges_by_seed.values()) + 1
checkpoint_steps = list(range(100, min(len(v) for v in cut_edges_by_seed.values()) + 1, 100))

tv_curve, w_curve, gr_curve = pairwise_max_at_checkpoints(
    cut_edges_by_seed, n_bins, checkpoint_steps
)
crossing_step, worst_curve = worst_seed_plan_tv_crossing(
    plan_ids_by_seed, pi, checkpoint_steps, 0.25
)
pairwise_steps = convergence_steps(
    tv_curve, w_curve, gr_curve, checkpoint_steps, 0.25, 0.1, 1.01
)

print(crossing_step)
print(pairwise_steps)
